In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
# Split Based on income
df["income_cat"] = pd.cut(df["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split

strat_train, strat_test = train_test_split(df, test_size=0.2, random_state=556,
                               stratify=df['income_cat'])


housing = strat_train.drop('income_cat', axis=1)
housing_labels = strat_train['median_house_value'].copy()
housing = housing.drop("median_house_value", axis=1)

# fix test set
housing_test = strat_test.drop('income_cat', axis=1)
housing_labels_test = strat_test['median_house_value'].copy()
housing_test = housing_test.drop("median_house_value", axis=1)


print(f"Train shape: {housing.shape}")
print(f"Train labels shape: {housing_labels.shape}")

print(f"Test shape: {housing_test.shape}")
print(f"Test labels shape: {housing_labels_test.shape}")


Train shape: (16512, 9)
Train labels shape: (16512,)
Test shape: (4128, 9)
Test labels shape: (4128,)


In [4]:
numerical_features = housing.select_dtypes(include=[np.number]).columns
categorical_features = housing.select_dtypes(include='object').columns

print(f"Numerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")

Numerical features: Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')
Categorical features: Index(['ocean_proximity'], dtype='object')


## Numerical Features

In [5]:
# Apply mean imputation
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
X_train_num = num_imputer.fit_transform(housing[numerical_features])
X_test_num = num_imputer.transform(housing_test[numerical_features])

In [6]:
# Apply Standard Scaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

## Categorical Features

In [7]:
# Apply Most Frequent Imputation
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat = cat_imputer.fit_transform(housing[categorical_features])
X_test_cat = cat_imputer.transform(housing_test[categorical_features])

In [8]:
# Apply OneHotEncoding
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
X_train_cat = encoder.fit_transform(X_train_cat)
X_test_cat = encoder.transform(X_test_cat)

In [9]:
print(f"Train numerical shape: {X_train_num.shape}")
print(f"Train categorical shape: {X_train_cat.shape}")

print(f"Test numerical shape: {X_test_num.shape}")
print(f"Test categorical shape: {X_test_cat.shape}")


Train numerical shape: (16512, 8)
Train categorical shape: (16512, 5)
Test numerical shape: (4128, 8)
Test categorical shape: (4128, 5)


## Combine the two feature sets

In [10]:
X_train = np.hstack((X_train_num, X_train_cat.toarray()))
X_test = np.hstack((X_test_num, X_test_cat.toarray()))

In [12]:
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}") 

Train shape: (16512, 13)
Test shape: (4128, 13)


## Train the model

In [13]:
# Train a model

from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, housing_labels)

# Evaluate the model
from sklearn.metrics import root_mean_squared_error

# Evaluate on test set
housing_predictions = lin_reg.predict(X_test)
lin_mse = root_mean_squared_error(housing_labels_test, housing_predictions)
lin_rmse = np.sqrt(lin_mse)

print(f"Test RMSE: {lin_rmse}")

Test RMSE: 260.4293440829138


## Make the prediction

In [15]:
# Create an example as a dictionary

example = {
    "longitude": -122.23,
    "latitude": 37.88,
    "housing_median_age": 41.0,
    "total_rooms": 880.0,
    "total_bedrooms": 129.0,
    "population": 322.0,
    "households": 126.0,
    "median_income": 8.3252,
    "ocean_proximity": "NEAR BAY"
}

example_df = pd.DataFrame([example])
example_df


# Apply the same transformations    
example_df_num = num_imputer.transform(example_df[numerical_features])
example_df_num = scaler.transform(example_df_num)

example_df_cat = cat_imputer.transform(example_df[categorical_features])
example_df_cat = encoder.transform(example_df_cat)

example_df = np.hstack((example_df_num, example_df_cat.toarray()))

# # Predict
example_prediction = lin_reg.predict(example_df)
example_prediction

array([408636.21021764])

## Save the model

In [17]:
import joblib

joblib.dump(lin_reg, 'linear_regression.pkl')
joblib.dump(num_imputer, 'num_imputer.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(cat_imputer, 'cat_imputer.pkl')
joblib.dump(encoder, 'encoder.pkl')

['encoder.pkl']

In [3]:
import pandas as pd
import numpy as np
import joblib

example = {
    "longitude": -122.23,
    "latitude": 37.88,
    "housing_median_age": 41.0,
    "total_rooms": 880.0,
    "total_bedrooms": 129.0,
    "population": 322.0,
    "households": 126.0,
    "median_income": 8.3252,
    "ocean_proximity": "NEAR BAY"
}

example_df = pd.DataFrame([example])

numerical_features = example_df.select_dtypes(include=[np.number]).columns
categorical_features = example_df.select_dtypes(include='object').columns

# Load the model
lin_reg = joblib.load('linear_regression.pkl')
num_imputer = joblib.load('num_imputer.pkl')
scaler = joblib.load('scaler.pkl')
cat_imputer = joblib.load('cat_imputer.pkl')
encoder = joblib.load('encoder.pkl')

example_df_num = num_imputer.transform(example_df[numerical_features])
example_df_num = scaler.transform(example_df_num)

example_df_cat = cat_imputer.transform(example_df[categorical_features])
example_df_cat = encoder.transform(example_df_cat)

example_df = np.hstack((example_df_num, example_df_cat.toarray()))

# Predict
example_prediction = lin_reg.predict(example_df)
example_prediction

array([408636.21021764])